<a href="https://colab.research.google.com/github/nikadekrahayuwindari-rgb/tugas-Ni-Kadek-Rahayu-Windari-data-/blob/main/Tugas%20RTM%201%3A%20Akuisisi%20Data%20Warisan%20Budaya%20Baliakuisisi/_wikipedia__ni_kadek_rahayu_windari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas RTM 1: Akuisisi Data Warisan Budaya Bali
### Program Studi Sistem Komputer / Teknologi Informasi — INSTIKI

---

**Identitas Pengembang:**
* **Nama:** Ni Kadek Rahayu Windari
* **NIM:** 2501010176
* **Mata Kuliah:** Akuisisi dan Manajemen Data (BIFP-243)

---

### Deskripsi Proyek
Notebook ini dirancang untuk melakukan akuisisi data artikel bertema budaya Bali secara terprogram dari MediaWiki Action API (Wikipedia Bahasa Indonesia). Seluruh proses dirancang dengan memperhatikan etika pengambilan data (*web scraping/crawling etiquette*), kepatuhan terhadap lisensi data, pembatasan beban server (*rate limiting*), serta penanganan galat secara otomatis.

### Alur Kerja & Organisasi Kode
1. **Setup Awal & Struktur Folder:** Menyiapkan konfigurasi identitas mahasiswa dan merancang repositori lokal (`rtm1_2501010176/`).
2. **Utilitas API dengan Mekanisme Fail-Safe:** Membuat fungsi request kustom yang mendukung pembatasan *maxlag*, penanganan jeda, serta *backoff* eksponensial.
3. **Validasi Kepatuhan & Etika:** Memeriksa batasan `robots.txt` dan mampir ke dokumentasi resmi kebijakan penggunaan API Wikimedia.
4. **Verifikasi Kategori Target:** Melakukan uji coba awal ketersediaan kategori budaya Bali untuk menghitung estimasi jumlah data.
5. **Paginasi Daftar Artikel:** Mengambil daftar pranala artikel unik dari kategori terpilih.
6. **Ekstraksi Teks Intro & Metadata:** Mengunduh potongan teks pengantar (*intro extracts*) dan riwayat ringkas pembaruan halaman secara berkala.
7. **Penyimpanan Dataset Mentah:** Menyimpan keluaran API secara utuh dalam format JSON asli serta CSV hasil ekstraksi awal.
8. **Penyusunan Metadata Provenance:** Menghitung ukuran berkas dan hash MD5 untuk menjaga keabsahan data asal.
9. **Dokumentasi & Checklist Etika:** Mengompilasi instrumen evaluasi diri (*checklist*) kepatuhan pengambilan data beserta berkas `README.md`.
10. **Integritas Sistem & Kompresi:** Memvalidasi kualitas data akhir sebelum dikemas ke dalam berkas arsip pengumpulan (.zip).

In [ ]:
# === 1. Konfigurasi Identitas & Struktur Folder ===
from pathlib import Path

# Identitas Mahasiswa
NIM = "2501010176"
NAMA = "Ni Kadek Rahayu Windari"
EMAIL_KAMPUS = "2501010176@instiki.ac.id"

# Pembuatan Struktur Folder Kerja
BASE_DIR = Path(f"rtm1_{NIM}")
sub_folders = ["notebooks", "data/raw", "docs"]

for sub in sub_folders:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"Folder proyek siap: {BASE_DIR}/")

Folder proyek siap: rtm1_2501010176/


In [ ]:
# === 2. Impor, Konfigurasi Request, dan Fungsi Pengambil Data ===
import time
import json
import re
import hashlib
import shutil
import zipfile
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta

# --- Konfigurasi API & Request ---
URL_API = "https://id.wikipedia.org/w/api.php"
JEDA = 2.0            # Jeda antar-request (syarat tugas: minimal 1.0 detik)
TIMEOUT = 15          # Batas waktu tunggu request (detik)
MAKS_HALAMAN = 5      # Batas maksimum paginasi per kategori
MAKS_PERCOBAAN = 5    # Batas maksimum percobaan ulang (retry)
JUMLAH_REQUEST = 0    # Penghitung total request terkirim
WITA = timezone(timedelta(hours=8))

# User-Agent resmi sesuai kebijakan Wikimedia
HEADERS = {
    "Accept": "application/json",
    "User-Agent": f"INSTIKI-BIFP243-Tugas1/1.0 ({NAMA} - {NIM}; {EMAIL_KAMPUS})",
}


# --- Fungsi Pembantu ---
def baca_retry_after(response, default):
    """Membaca nilai header 'Retry-After' jika terjadi rate limiting."""
    try:
        return max(float(response.headers.get("Retry-After", default)), 1.0)
    except ValueError:
        return float(default)


def ambil_dengan_retry(params):
    """Mengambil data dari MediaWiki API menggunakan penanganan kegagalan secara aman."""
    global JUMLAH_REQUEST
    params = {**params, "maxlag": 5}  # Rekomendasi Wikimedia untuk request non-interaktif
    jeda = JEDA

    for percobaan in range(1, MAKS_PERCOBAAN + 1):
        try:
            r = requests.get(URL_API, params=params, headers=HEADERS, timeout=TIMEOUT)
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[Percobaan {percobaan}] {type(e).__name__}, mencoba lagi dalam {jeda}s...")
            time.sleep(jeda)
            jeda *= 2
            continue

        JUMLAH_REQUEST += 1

        # Penanganan response sukses
        if r.status_code == 200:
            data = r.json()
            if "error" in data:
                if data["error"].get("code") == "maxlag":
                    tunggu = max(baca_retry_after(r, 5), 5)
                    print(f"[Percobaan {percobaan}] Server sibuk (maxlag), menunggu {tunggu}s...")
                    time.sleep(tunggu)
                    continue
                raise RuntimeError(f"Error dari API: {data['error']}")
            return data

        # Penanganan status limit (429) atau server sibuk (503)
        if r.status_code in (429, 503):
            tunggu = baca_retry_after(r, jeda)
            print(f"[Percobaan {percobaan}] Status {r.status_code} (Rate Limit/Sibuk), menunggu {tunggu}s...")
            time.sleep(tunggu)
        elif 500 <= r.status_code < 600:
            print(f"[Percobaan {percobaan}] Status {r.status_code} (Error Server), menunggu {jeda}s...")
            time.sleep(jeda)
        else:
            raise RuntimeError(f"Error HTTP: {r.status_code} {r.reason}")

        jeda *= 2

    raise RuntimeError(f"Gagal mengambil data setelah {MAKS_PERCOBAAN} percobaan.")


# --- Inisialisasi ---
print("Fungsi siap.")
print("User-Agent:", HEADERS["User-Agent"])

Fungsi siap.
User-Agent: INSTIKI-BIFP243-Tugas1/1.0 (Ni Kadek Rahayu Windari - 2501010176; 2501010176@instiki.ac.id)


In [ ]:
# === 3. Pemeriksaan robots.txt & Ketentuan API ===
from urllib.robotparser import RobotFileParser

# --- Konfigurasi Sumber Aturan ---
URL_ROBOTS = "https://id.wikipedia.org/robots.txt"
URL_KETENTUAN = {
    "API Etiquette": "https://www.mediawiki.org/wiki/API:Etiquette",
    "User-Agent Policy": "https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy",
    "Maxlag": "https://www.mediawiki.org/wiki/Manual:Maxlag_parameter",
    "Lisensi teks (CC BY-SA 4.0)": "https://creativecommons.org/licenses/by-sa/4.0/",
}

# --- Proses Pemeriksaan ---
status_robots, boleh_api = None, None
teks_robots = ""

try:
    r = requests.get(URL_ROBOTS, headers=HEADERS, timeout=TIMEOUT)
    status_robots = r.status_code
    JUMLAH_REQUEST += 1

    if r.status_code == 200:
        teks_robots = r.text
        rp = RobotFileParser()
        rp.parse(teks_robots.splitlines())
        boleh_api = rp.can_fetch(HEADERS["User-Agent"], URL_API)
except requests.RequestException as e:
    print("Gagal membaca robots.txt:", type(e).__name__)

time.sleep(JEDA)

# --- Pencatatan Waktu & Penyimpanan Bukti ---
waktu_cek = datetime.now(WITA).strftime("%Y-%m-%d %H:%M:%S WITA")
(BASE_DIR / "docs" / "robots_id_wikipedia.txt").write_text(teks_robots, encoding="utf-8")

bukti = {
    "url_robots": URL_ROBOTS,
    "status_http": status_robots,
    "waktu_cek": waktu_cek,
    "can_fetch_api_php": boleh_api,
    "catatan": (
        "robots.txt ditujukan untuk crawler halaman web. Pengambilan data dilakukan lewat "
        "API resmi MediaWiki yang mengikuti API Etiquette dan User-Agent Policy Wikimedia "
        "(User-Agent jujur, maxlag, jeda, dan Retry-After dihormati)."
    ),
    "ketentuan_api": URL_KETENTUAN,
}

(BASE_DIR / "docs" / "bukti_robots_ketentuan.json").write_text(
    json.dumps(bukti, indent=2, ensure_ascii=False), encoding="utf-8"
)

# --- Tampilan Hasil ---
print(f"Status robots.txt     : {status_robots}")
print(f"can_fetch ke api.php  : {boleh_api}")
print("\nBukti berhasil disimpan di:")
print(f" - {BASE_DIR}/docs/bukti_robots_ketentuan.json")
print(f" - {BASE_DIR}/docs/robots_id_wikipedia.txt")

Status robots.txt     : 200
can_fetch ke api.php  : False

Bukti berhasil disimpan di:
 - rtm1_2501010176/docs/bukti_robots_ketentuan.json
 - rtm1_2501010176/docs/robots_id_wikipedia.txt


In [ ]:
# === 4. Cek Kategori Sebelum Mengambil Data ===
KATEGORI = [
    "Kategori:Budaya_Bali",
    "Kategori:Pura_di_Bali",
    "Kategori:Tarian_Bali",
    "Kategori:Musik_Bali",
    "Kategori:Bangunan_dan_struktur_di_Bali",
]

total_perkiraan = 0
print(f"{'Nama Kategori':45s} | {'Ada?':5s} | {'Artikel Langsung':16s}")
print("-" * 75)

for kat in KATEGORI:
    params = {
        "action": "query",
        "prop": "categoryinfo",
        "titles": kat,
        "format": "json"
    }

    res = ambil_dengan_retry(params)
    hal = list(res["query"]["pages"].values())[0]
    ada = "missing" not in hal
    n_artikel = hal.get("categoryinfo", {}).get("pages", 0)
    total_perkiraan += n_artikel

    print(f"{kat:45s} | {str(ada):5s} | {n_artikel:16d}")
    time.sleep(JEDA)

print("-" * 75)
print(f"Perkiraan total sebelum digabung (mungkin ada duplikat): {total_perkiraan}")

if total_perkiraan < 100:
    print("\nPERINGATAN: Jumlah artikel kurang dari 100!")
    print("Silakan tambahkan atau ganti kategori pada daftar KATEGORI di atas.")

Nama Kategori                                 | Ada?  | Artikel Langsung
---------------------------------------------------------------------------
Kategori:Budaya_Bali                          | True  |              139
Kategori:Pura_di_Bali                         | True  |               38
Kategori:Tarian_Bali                          | False |                1
Kategori:Musik_Bali                           | True  |                8
Kategori:Bangunan_dan_struktur_di_Bali        | True  |               16
---------------------------------------------------------------------------
Perkiraan total sebelum digabung (mungkin ada duplikat): 202


In [ ]:
# === 5. Ambil Daftar Artikel Per Kategori ===
waktu_mulai = datetime.now(WITA)
halaman = {}        # Menyimpan page_id -> daftar kategori asal
raw_kategori = []   # Menyimpan respon API mentah untuk validasi

print(f"{'Kategori':30s} | {'Halaman':8s} | {'Total Artikel Unik Terkumpul'}")
print("-" * 70)

for kat in KATEGORI:
    nama_kat = kat.replace("Kategori:", "").replace("_", " ")
    lanjut = None

    for hal in range(1, MAKS_HALAMAN + 1):
        params = {
            "action": "query",
            "list": "categorymembers",
            "cmtitle": kat,
            "cmtype": "page",
            "cmlimit": 50,
            "format": "json",
        }
        if lanjut:
            params["cmcontinue"] = lanjut

        res = ambil_dengan_retry(params)
        raw_kategori.append(res)

        for item in res.get("query", {}).get("categorymembers", []):
            daftar = halaman.setdefault(item["pageid"], [])
            if nama_kat not in daftar:
                daftar.append(nama_kat)

        print(f"{nama_kat:30s} | Hal {hal:4d} | {len(halaman):d}")
        time.sleep(JEDA)

        lanjut = res.get("continue", {}).get("cmcontinue")
        if not lanjut:
            break
    else:
        print(f"  (Batas {MAKS_HALAMAN} halaman tercapai untuk {nama_kat}; mungkin masih ada artikel lain)")

print("-" * 70)
print(f"Selesai. Total artikel unik terkumpul: {len(halaman)}")

Kategori                       | Halaman  | Total Artikel Unik Terkumpul
----------------------------------------------------------------------
Budaya Bali                    | Hal    1 | 50
Budaya Bali                    | Hal    2 | 100
Budaya Bali                    | Hal    3 | 139
Pura di Bali                   | Hal    1 | 177
Tarian Bali                    | Hal    1 | 178
Musik Bali                     | Hal    1 | 185
Bangunan dan struktur di Bali  | Hal    1 | 198
----------------------------------------------------------------------
Selesai. Total artikel unik terkumpul: 198


In [ ]:
# === 6. Ambil Detail Tiap Artikel ===
ids = list(halaman)
raw_detail = []
baris = []

print(f"Memulai ekstraksi detail untuk {len(ids)} artikel...")
print(f"{'Kemajuan':15s} | {'Jumlah Berhasil':15s} | {'Status'}")
print("-" * 55)

for i in range(0, len(ids), 20):
    potongan = ids[i:i + 20]
    params = {
        "action": "query",
        "prop": "extracts|info",
        "exintro": 1,
        "explaintext": 1,
        "inprop": "url",
        "pageids": "|".join(str(p) for p in potongan),
        "format": "json",
    }

    pages = {}
    # Batas aman untuk menangani kelanjutan (continue) jika ada
    for _ in range(5):
        res = ambil_dengan_retry(params)
        raw_detail.append(res)
        for pid, isi in res.get("query", {}).get("pages", {}).items():
            pages.setdefault(pid, {}).update(isi)

        time.sleep(JEDA)
        if "continue" in res:
            params.update(res["continue"])
        else:
            break

    for pid_str, isi in pages.items():
        if "missing" in isi:
            continue
        pid = int(pid_str)
        baris.append({
            "page_id": pid,
            "judul_artikel": isi.get("title", ""),
            "kategori_budaya": "; ".join(halaman.get(pid, [])),
            "ringkasan": isi.get("extract", ""),
            "panjang_halaman": isi.get("length", 0),
            "terakhir_diperbarui": isi.get("touched", ""),
            "id_revisi_terakhir": isi.get("lastrevid", 0),
            "url_sumber": isi.get("fullurl", f"https://id.wikipedia.org/?curid={pid}"),
        })

    persen = (len(baris) / len(ids)) * 100
    print(f"Sesi {i//20 + 1:3d}        | {len(baris):4d} / {len(ids):4d}    | Berhasil ({persen:.1f}%)")

print("-" * 55)
print(f"Selesai. Total detail berhasil diekstraksi: {len(baris)} artikel.")

Memulai ekstraksi detail untuk 198 artikel...
Kemajuan        | Jumlah Berhasil | Status
-------------------------------------------------------
Sesi   1        |   20 /  198    | Berhasil (10.1%)
Sesi   2        |   40 /  198    | Berhasil (20.2%)
Sesi   3        |   60 /  198    | Berhasil (30.3%)
Sesi   4        |   80 /  198    | Berhasil (40.4%)
Sesi   5        |  100 /  198    | Berhasil (50.5%)
Sesi   6        |  120 /  198    | Berhasil (60.6%)
Sesi   7        |  140 /  198    | Berhasil (70.7%)
Sesi   8        |  160 /  198    | Berhasil (80.8%)
Sesi   9        |  180 /  198    | Berhasil (90.9%)
Sesi  10        |  198 /  198    | Berhasil (100.0%)
-------------------------------------------------------
Selesai. Total detail berhasil diekstraksi: 198 artikel.


In [ ]:
# === 7. Penyimpanan Data Mentah ke data/raw/ ===
waktu_selesai = datetime.now(WITA)
cap_tgl = waktu_mulai.strftime("%Y%m%d")

file_json = BASE_DIR / "data" / "raw" / f"wikipedia_budaya_{cap_tgl}.json"
file_csv = BASE_DIR / "data" / "raw" / f"wikipedia_budaya_{cap_tgl}.csv"

# Menyimpan struktur JSON asli yang dikembalikan oleh API (raw data)
file_json.write_text(
    json.dumps({"categorymembers": raw_kategori, "detail_halaman": raw_detail},
               indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Menyimpan struktur CSV hasil tabelarisasi awal tanpa manipulasi berlebih
df = pd.DataFrame(baris)
df.to_csv(file_csv, index=False, encoding="utf-8")

print(f"Dataset berhasil disimpan:")
print(f"  - CSV: {file_csv} ({len(df)} baris, {len(df.columns)} kolom)")
print(f"  - JSON: {file_json}")

# Menampilkan pratinjau data teratas secara rapi
df.head()

Dataset berhasil disimpan:
  - CSV: rtm1_2501010176/data/raw/wikipedia_budaya_20261115.csv (198 baris, 8 kolom)
  - JSON: rtm1_2501010176/data/raw/wikipedia_budaya_20261115.json


,page_id,judul_artikel,kategori_budaya,ringkasan,panjang_halaman,terakhir_diperbarui,id_revisi_terakhir,url_sumber
0,4985,Astawara,Budaya Bali,Astawara adalah nama dari sebuah pekan atau mi...,460,2026-11-12T09:30:13Z,26224862,https://id.wikipedia.org/wiki/Astawara
1,74300,Ayam betutu,Budaya Bali,Ayam Betutu adalah makanan tradisional khas Ba...,6995,2026-11-12T13:54:32Z,30006409,https://id.wikipedia.org/wiki/Ayam_betutu
2,74883,Joged Bumbung,Budaya Bali,Joged Bumbung merupakan salah satu tarian trad...,5054,2026-11-12T13:54:32Z,29801860,https://id.wikipedia.org/wiki/Joged_Bumbung
3,143082,Arja,Budaya Bali,"Arja adalah semacam opera khas Bali, merupakan...",5934,2026-10-14T05:45:20Z,28929565,https://id.wikipedia.org/wiki/Arja
4,277626,Aksara Bali,Budaya Bali,"Aksara Bali, juga dikenal sebagai Hanacaraka, ...",107219,2026-11-12T13:54:38Z,30123959,https://id.wikipedia.org/wiki/Aksara_Bali


In [ ]:
# === 8. Pembuatan Metadata & Provenance (docs/provenance.json) ===
def info_berkas(path):
    """Mendapatkan ukuran berkas dalam byte dan hash MD5."""
    isi = path.read_bytes()
    return len(isi), hashlib.md5(isi).hexdigest()

ukuran_csv, md5_csv = info_berkas(file_csv)
ukuran_json, md5_json = info_berkas(file_json)
waktu_akuisisi = waktu_mulai.strftime("%Y-%m-%d %H:%M:%S WITA")

pengambil = {
    "nama": NAMA,
    "nim": NIM,
    "email": EMAIL_KAMPUS
}

umum = {
    "sumber": "Wikipedia Bahasa Indonesia (MediaWiki Action API), Wikimedia Foundation",
    "url": URL_API,
    "lisensi": "CC BY-SA 4.0 (atribusi: Wikipedia Bahasa Indonesia)",
    "waktu_akuisisi": waktu_akuisisi,
    "waktu_akuisisi_utc": waktu_mulai.astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
    "pengambil": pengambil,
}

provenance = {
    "berkas": [
        {
            "nama_berkas": file_csv.name,
            "ukuran_byte": ukuran_csv,
            "hash_md5": md5_csv,
            "jenis": "tabel hasil ekstraksi (tanpa cleaning)",
            "jumlah_record": len(df),
            "jumlah_kolom": len(df.columns),
            **umum
        },
        {
            "nama_berkas": file_json.name,
            "ukuran_byte": ukuran_json,
            "hash_md5": md5_json,
            "jenis": "respons API mentah",
            **umum
        },
    ],
    "metode": {
        "jenis": "API resmi, tanpa API key",
        "daftar_artikel": {
            "action": "query",
            "list": "categorymembers",
            "cmtype": "page",
            "cmlimit": 50,
            "kategori": KATEGORI
        },
        "detail_artikel": {
            "action": "query",
            "prop": "extracts|info",
            "exintro": 1,
            "explaintext": 1,
            "inprop": "url",
            "pageids": "20 ID per request"
        },
        "jumlah_request_total": JUMLAH_REQUEST,
        "jeda_detik": JEDA,
        "timeout_detik": TIMEOUT,
        "maks_halaman_per_kategori": MAKS_HALAMAN,
    },
    "waktu_selesai": waktu_selesai.strftime("%Y-%m-%d %H:%M:%S WITA"),
}

(BASE_DIR / "docs" / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Provenance JSON berhasil dikompilasi:")
print(f"  - Lokasi: {BASE_DIR}/docs/provenance.json")
print(f"  - MD5 CSV : {md5_csv}")
print(f"  - MD5 JSON: {md5_json}")

Provenance JSON berhasil dikompilasi:
  - Lokasi: rtm1_2501010176/docs/provenance.json
  - MD5 CSV : afc4bb70bb18afcf8d0d246f54e8100e
  - MD5 JSON: 349b2aa2af1dec78579ae18f2793c29a


In [ ]:
# === 9. Pembuatan Data Ethics Checklist & README.md ===
hasil_robots = ("tidak dapat dibaca" if boleh_api is None
                else ("diizinkan" if boleh_api else "tidak diizinkan untuk crawler"))

ceklis = [
    (1, "Legalitas sumber data", "Ya",
     f"API resmi MediaWiki ({URL_API}) yang terbuka untuk publik, tanpa login."),
    (2, "Kepatuhan lisensi dan atribusi", "Ya",
     "Teks Wikipedia berlisensi CC BY-SA 4.0; atribusi ke Wikipedia Bahasa Indonesia ada di README dan provenance.json."),
    (3, "Kepatuhan robots.txt / ketentuan API", "Ya (via API resmi)",
     f"robots.txt diperiksa dengan RobotFileParser (hasil can_fetch api.php: {hasil_robots}), bukti di docs/bukti_robots_ketentuan.json. "
     "robots.txt ditujukan bagi crawler halaman web; akses terprogram dilakukan lewat API resmi yang mengikuti API Etiquette dan User-Agent Policy Wikimedia."),
    (4, "Identitas User-Agent jujur", "Ya",
     f"User-Agent memuat nama, NIM, dan email kampus: {HEADERS['User-Agent']}"),
    (5, "Penghormatan rate limit dan jeda", "Ya",
     f"Jeda {JEDA} detik antar-request, parameter maxlag=5, header Retry-After dihormati, exponential backoff untuk 429/5xx. Total request: {JUMLAH_REQUEST}."),
    (6, "Batas paginasi dan timeout", "Ya",
     f"Maksimum {MAKS_HALAMAN} halaman per kategori, timeout {TIMEOUT} detik, maksimum {MAKS_PERCOBAAN} percobaan per request."),
    (7, "Tidak mengambil data pribadi", "Ya",
     "Hanya metadata artikel dan paragraf pembuka; tidak mengambil nama editor, riwayat revisi, email, atau kontak. Teks adalah konten ensiklopedis publik; tidak ada data pribadi individu privat, kontak, atau akun media sosial."),
    (8, "Keamanan kredensial / API key", "Ya",
     "Tidak memakai API key; tidak ada kredensial di notebook."),
    (9, "Keutuhan data mentah dan provenance", "Ya",
     f"Berkas di data/raw/ tidak diubah setelah disimpan; MD5 CSV {md5_csv}, MD5 JSON {md5_json}."),
]

# Ekspor checklist ke CSV
pd.DataFrame(ceklis, columns=["no", "butir_pemeriksaan", "status", "bukti"]).to_csv(
    BASE_DIR / "docs" / "data_ethics_checklist.csv", index=False, encoding="utf-8"
)

# Formulasi dokumen README.md
readme = f"""# Katalog Warisan Budaya Bali (Wikipedia Bahasa Indonesia API)

Tugas Praktik Individu RTM Tugas 1, Akuisisi dan Manajemen Data (BIFP-243)
**Nama:** {NAMA} | **NIM:** {NIM}

## Tujuan dan alasan pemilihan sumber
Mengumpulkan katalog artikel budaya Bali (budaya umum, pura, tari, musik, bangunan/struktur) sebagai dataset mentah untuk RTM Tugas 2. Wikipedia Bahasa Indonesia dipilih karena relevan dengan sektor budaya dan pariwisata, menyediakan API resmi tanpa API key, berlisensi terbuka, dan artikelnya terstruktur dalam kategori sehingga cakupannya bisa dijelaskan dan diulang.

## 1. Sumber data
* **Nama:** Wikipedia Bahasa Indonesia (MediaWiki Action API)
* **URL API:** `{URL_API}`
* **URL referensi:** https://id.wikipedia.org/wiki/Kategori:Budaya_Bali
* **Penerbit:** Wikimedia Foundation
* **Atribusi:** Data bersumber dari Wikipedia Bahasa Indonesia, lisensi CC BY-SA 4.0.

## 2. Tanggal dan jam akses
`{waktu_akuisisi}` (selesai `{waktu_selesai.strftime("%Y-%m-%d %H:%M:%S WITA")}`). Sama dengan `docs/provenance.json`.

## 3. Metode akuisisi
* **Jenis:** API resmi, tanpa API key.
* **Daftar artikel:** `action=query`, `list=categorymembers`, `cmtitle` = {", ".join(f"`{k}`" for k in KATEGORI)}, `cmtype=page`, `cmlimit=50`, `format=json`, paginasi lewat `cmcontinue`.
* **Detail artikel:** `action=query`, `prop=extracts|info`, `exintro=1`, `explaintext=1`, `inprop=url`, `pageids` (20 ID per request), `format=json`.
* **Pengaturan request:** `timeout={TIMEOUT}`, User-Agent dengan email kampus, maksimum {MAKS_HALAMAN} halaman per kategori, jeda {JEDA} detik, `maxlag=5`, retry dengan exponential backoff, dan header `Retry-After` dihormati. Total request yang dikirim: **{JUMLAH_REQUEST}**.

## 4. Batasan etika
* **robots.txt:** diperiksa dengan `RobotFileParser`, hasil `can_fetch` untuk `/w/api.php`: **{hasil_robots}**. robots.txt ditujukan untuk crawler halaman web; pengambilan data dilakukan lewat API resmi. Bukti: `docs/bukti_robots_ketentuan.json` dan `docs/robots_id_wikipedia.txt`.
* **Ketentuan API:** mengikuti API Etiquette dan User-Agent Policy Wikimedia (User-Agent jujur, `maxlag`, jeda, `Retry-After`).
* **Lisensi:** CC BY-SA 4.0, atribusi dicantumkan di atas.
* **Data yang sengaja tidak diambil:** nama editor/kontributor, riwayat revisi, isi artikel penuh (hanya paragraf pembuka), gambar, dan subkategori.
* **Data pribadi:** tidak ada data pribadi perorangan yang diambil. Ringkasan artikel adalah konten ensiklopedis publik (dapat menyebut tokoh publik atau sejarah), bukan data pribadi individu privat; tidak ada email, telepon, atau akun media sosial.
* **Keterbatasan:** hanya artikel yang langsung terdaftar di lima kategori target (subkategori tidak ditelusuri). Artikel yang ada di lebih dari satu kategori ditulis satu baris dengan kategori digabung `;`.
* **Data mentah:** CSV dan JSON di `data/raw/` tidak diubah setelah disimpan; pembersihan dilakukan di RTM Tugas 2.

## 5. Penjelasan kolom (`data/raw/{file_csv.name}`)
| Kolom | Tipe | Arti | Satuan |
| :--- | :--- | :--- | :--- |
| `page_id` | bilangan bulat | ID unik artikel di Wikipedia | - |
| `judul_artikel` | teks | Judul artikel | - |
| `kategori_budaya` | teks | Kategori asal artikel (dipisah `;` bila lebih dari satu) | - |
| `ringkasan` | teks | Paragraf pembuka artikel (teks utuh, tidak dipotong) | - |
| `panjang_halaman` | bilangan bulat | Ukuran konten halaman | byte |
| `terakhir_diperbarui` | tanggal-waktu | Waktu terakhir halaman diperbarui/di-refresh server (`touched`; bisa karena suntingan atau perubahan templat), ISO 8601 | UTC |
| `id_revisi_terakhir` | bilangan bulat | ID revisi terbaru artikel | - |
| `url_sumber` | teks | Tautan langsung ke artikel | - |

Jumlah record: **{len(df)}** baris, **{len(df.columns)}** kolom. Berkas pendukung: `{file_json.name}` (respons API mentah).

## 6. Cara menjalankan ulang
1. `pip install requests pandas`
2. Buka `notebooks/akuisisi_wikipedia.ipynb`, pilih **Restart & Run All**.
3. Hasil tersimpan di `data/raw/`, `docs/provenance.json`, `docs/data_ethics_checklist.csv`, dan README ini dibuat ulang.
"""

# Menyimpan berkas README.md
(BASE_DIR / "README.md").write_text(readme, encoding="utf-8")

print("Pembuatan dokumen penunjang selesai:")
print(f"  - Checklist Etika : {BASE_DIR}/docs/data_ethics_checklist.csv")
print(f"  - Berkas README    : {BASE_DIR}/README.md")

Pembuatan dokumen penunjang selesai:
  - Checklist Etika : rtm1_2501010176/docs/data_ethics_checklist.csv
  - Berkas README    : rtm1_2501010176/README.md


In [ ]:
# === 10. Validasi Akhir & Ekspor Arsip ZIP ===
NAMA_NOTEBOOK = "akuisisi_wikipedia.ipynb"

print("Memulai verifikasi integritas data...")
print("-" * 50)

# Validasi Kriteria Tugas
assert len(df) >= 100, f"Peringatan: Jumlah record ({len(df)}) kurang dari batas minimal 100!"
assert df.shape[1] >= 5, f"Peringatan: Jumlah kolom ({df.shape[1]}) kurang dari batas minimal 5!"

duplikat_id = df["page_id"].duplicated().sum()
ringkasan_kosong = (df["ringkasan"].str.strip() == "").sum()

print(f"  - Validasi Rekor  : OK ({len(df)} baris terdeteksi)")
print(f"  - Validasi Kolom  : OK ({df.shape[1]} kolom terdeteksi)")
print(f"  - Duplikasi ID    : {duplikat_id} data")
print(f"  - Ringkasan Kosong: {ringkasan_kosong} baris")
print(f"  - Total API Calls : {JUMLAH_REQUEST} request terkirim")

# Memastikan tidak ada kebocoran data pribadi sensitif (seperti email atau nomor telepon)
pola_sensitif = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+|(?:\+62|\b08)\d{8,}")
temukan_sensitif = int(df["ringkasan"].str.contains(pola_sensitif).sum())
print(f"  - Data Sensitif   : {temukan_sensitif} terdeteksi")
print("-" * 50)

# Menyalin notebook ke folder kerja jika berkas ditemukan
if Path(NAMA_NOTEBOOK).exists():
    shutil.copy(NAMA_NOTEBOOK, BASE_DIR / "notebooks" / NAMA_NOTEBOOK)
    print(f"Notebook '{NAMA_NOTEBOOK}' berhasil disalin ke {BASE_DIR}/notebooks/")
else:
    print(f"[!] Catatan: Berkas '{NAMA_NOTEBOOK}' tidak ditemukan di direktori lokal Colab.\n"
          f"    Silakan unduh notebook ini secara manual (File > Download > .ipynb) dan masukkan\n"
          f"    ke dalam folder '{BASE_DIR}/notebooks/' sebelum melakukan pengumpulan.")

# Membuat arsip ZIP untuk pengumpulan tugas
shutil.make_archive(f"rtm1_{NIM}", "zip", ".", f"rtm1_{NIM}")

print(f"\nArsip ZIP berhasil dibuat: rtm1_{NIM}.zip")
print("Daftar berkas di dalam arsip:")
for berkas_arsip in zipfile.ZipFile(f"rtm1_{NIM}.zip").namelist():
    print(f"  [+] {berkas_arsip}")

# Fungsi unduh otomatis untuk platform Google Colab
try:
    from google.colab import files
    files.download(f"rtm1_{NIM}.zip")
except ImportError:
    pass

Memulai verifikasi integritas data...
--------------------------------------------------
  - Validasi Rekor  : OK (198 baris terdeteksi)
  - Validasi Kolom  : OK (8 kolom terdeteksi)
  - Duplikasi ID    : 0 data
  - Ringkasan Kosong: 2 baris
  - Total API Calls : 23 request terkirim
  - Data Sensitif   : 0 terdeteksi
--------------------------------------------------
[!] Catatan: Berkas 'akuisisi_wikipedia.ipynb' tidak ditemukan di direktori lokal Colab.
    Silakan unduh notebook ini secara manual (File > Download > .ipynb) dan masukkan
    ke dalam folder 'rtm1_2501010176/notebooks/' sebelum melakukan pengumpulan.

Arsip ZIP berhasil dibuat: rtm1_2501010176.zip
Daftar berkas di dalam arsip:
  [+] rtm1_2501010176/
  [+] rtm1_2501010176/data/
  [+] rtm1_2501010176/docs/
  [+] rtm1_2501010176/notebooks/
  [+] rtm1_2501010176/README.md
  [+] rtm1_2501010176/data/raw/
  [+] rtm1_2501010176/data/raw/wikipedia_budaya_20261115.json
  [+] rtm1_2501010176/data/raw/wikipedia_budaya_20261115.csv

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>